# 밀폐 공간 전술 정찰 및 구조를 위한 온디바이스 임베디드 AI 드론
## 3-OFF + MAP: GPS-Denied 실내 객체 탐지 모델 학습

이 노트북은 **Google Colab** 환경에서 실행되도록 최적화되어 있습니다.
런타임 유형을 반드시 **GPU**로 설정하고 진행해 주세요.

In [ ]:
# 1. 환경 설정 및 GPU 확인
!pip install ultralytics
from ultralytics import YOLO
import torch
import os

print(f"GPU 확인: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"현재 사용 중인 GPU: {torch.cuda.get_device_name(0)}")

## 2. 대용량 데이터셋 고속 다운로드
Colab의 초고속 인터넷망을 활용하여 1GB 이상의 데이터셋을 빠르게 다운로드합니다.

In [ ]:
# YOLOv8은 학습 시 coco8.yaml 등 기본 yaml 파일을 지정하면 자동으로 샘플 데이터셋을 다운로드합니다.
# (기존 수동 다운로드 및 압축 해제 코드로 인한 에러 방지를 위해 내용 변경)
print("데이터셋 다운로드 준비 완료! (아래 학습 코드에서 자동 다운로드됩니다)")

In [ ]:
# D-Fire 데이터셋 다운로드 (Kaggle)
# 주의: 이 셀을 실행하기 전에 Kaggle API 키(kaggle.json)를 Colab 환경에 업로드해야 합니다.
print("D-Fire 데이터셋 다운로드는 Kaggle API 설정이 필요합니다.")
print("kaggle.json 파일을 업로드하신 후 아래 코드를 주석 해제하여 실행하세요.")

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d dataclusterlabs/fire-and-smoke-dataset -p /content/datasets/dfire --unzip

## 3. 모델 학습
다운로드한 데이터를 기반으로 YOLOv8n 모델 전이학습을 시작합니다.

In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data='coco8.yaml', # 빠른 테스트를 위한 미니 COCO 데이터셋. (실제 학습 시에는 D-Fire 등 커스텀 yaml 파일 경로 입력)
    epochs=5, # 테스트용이므로 5 에포크만 진행. (실제 학습 시 50 이상 권장)
    imgsz=640, 
    batch=16, 
    optimizer='AdamW', 
    lr0=0.001, 
    patience=10, 
    # device=0, # <-- 로컬(내 PC)에서 돌릴 때 에러가 나지 않도록 주석 처리했습니다. 코랩에서는 자동으로 GPU를 잡습니다.
    seed=42
)

In [ ]:
from ultralytics import YOLO
import os

# 셀만 단독으로 실행하거나 커널이 재시작되더라도, 학습된 모델을 제대로 불러올 수 있도록 안정화 처리
if os.path.exists('runs/detect/train/weights/best.pt'):
    model = YOLO('runs/detect/train/weights/best.pt')
    print("학습된 베스트 모델 로드 완료.")
else:
    model = YOLO('yolov8n.pt')
    print("학습된 모델 경로(runs/...)가 없어 기본 모델을 로드합니다.")

print("Jetson 보드 배포를 위한 ONNX 변환을 시작합니다...")
model.export(format='onnx')